# Error Handling & Recovery

*Notebook 06*

Tools fail. APIs time out. Data is missing.

The SDK turns function-tool exceptions into error results by default.

You decide whether to return better guidance, retry, or stop the run.

---

## 🔧 Setup

In [ ]:
import time
import random
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

Error handling is the difference between a demo and a production tool.

---

## 💥 Part 1: Letting an Exception Stop the Run

By default, the SDK returns a standard tool-error result to the model.

Set `failure_error_function=None` to stop the run on a tool failure: the SDK raises `UserError` chaining the original exception.

This demo makes that stop path visible.

In [ ]:
# Simulated database: only some records exist
PRODUCT_DB = {
    "P001": {"name": "Wireless Headphones", "price": 149.99, "stock": 23},
    "P002": {"name": "Phone Case", "price": 19.99, "stock": 0},
    "P003": {"name": "Laptop Stand", "price": 49.99, "stock": 7}
}


@function_tool(failure_error_function=None)  # Demo only: lets a tool exception stop the run
def lookup_product_fragile(product_id: str) -> str:
    """Look up product details by ID."""
    product = PRODUCT_DB[product_id]  # KeyError if product not found
    return f"{product['name']}: ${product['price']:.2f} ({product['stock']} in stock)"


fragile_agent = Agent(
    name="FragileAgent",
    instructions="Look up product information using the available tool.",
    model=MODEL,
    tools=[lookup_product_fragile]
)

# --------------------------------------------------------------
print("✅ Fragile agent ready, intentionally broken")

#### What Happens on a Bad Input

In [ ]:
print("💥 UNHANDLED EXCEPTION DEMO")
print("=" * 60)

try:

    result = await Runner.run(fragile_agent, input="What's the price of product P999?")

    print(result.final_output)

except Exception as e:
    print(f"❌ Run failed: {type(e).__name__}: {e}")
    print("   The entire agent run crashed. User gets nothing")

print("=" * 60)

---

## 🛡️ Part 2: Handling Errors Inside the Tool

Handle expected failures inside the tool when the model needs domain-specific guidance.

For a missing product, return the valid IDs instead of exposing `KeyError` details.

In [ ]:
@function_tool
def lookup_product(product_id: str) -> str:
    """Look up product details by ID."""
    print(f"  🔧 [tool] lookup_product called with {product_id}")
    try:
        product = PRODUCT_DB[product_id]
        stock_status = "in stock" if product["stock"] > 0 else "out of stock"
        detail = f"${product['price']:.2f} ({product['stock']} units {stock_status})"
        return f"{product['name']}: {detail}"
    except KeyError:
        available = ", ".join(PRODUCT_DB.keys())
        return f"Product '{product_id}' not found. Available products: {available}"
    except Exception as error:
        print(f"  [log] lookup_product failed: {type(error).__name__}: {error}")
        return "Unable to look up products right now. Please try again later."


robust_instructions = (
    "Look up product information using the available tool.\n"
    "Report only the lookup result.\n"
    "If the product is missing, name the available product IDs.\n"
    "Do not offer actions beyond the available tool."
)

robust_agent = Agent(
    name="RobustAgent",
    instructions=robust_instructions,
    model=MODEL,
    tools=[lookup_product]
)

# --------------------------------------------------------------
print("✅ Robust agent ready")

#### Valid Product vs Missing Product

In [ ]:
print("🛡️ ROBUST TOOL DEMO")
print("=" * 60)

for query in [
    "What's the price of product P001?",
    "What's the price of product P999?"
]:
    print(f"\n🙋 {query}")

    result = await Runner.run(robust_agent, input=query)

    print(f"🤖 {result.final_output}")

print("\n" + "=" * 60)

---

## 🔁 Part 3: Retry Pattern

Some failures are transient: a flaky API, a rate limit, a momentary network issue.

For these, retry with a limit rather than failing immediately.

In [ ]:
# Simulates an unreliable external API: fails 60% of the time
def unreliable_api_call(query: str) -> str:
    """Simulates a flaky external service."""
    if random.random() < 0.6:
        raise ConnectionError("Service temporarily unavailable")
    return f"API result for: {query}"


@function_tool
def fetch_with_retry(query: str) -> str:
    """Fetch data from an external service with automatic retry."""
    max_attempts = 3
    wait_seconds = 1

    for attempt in range(1, max_attempts + 1):
        try:
            result = unreliable_api_call(query)
            print(f"    ✅ Succeeded on attempt {attempt}")
            return result
        except ConnectionError as e:
            print(f"    ⚠️  Attempt {attempt} failed: {e}")
            if attempt < max_attempts:
                time.sleep(wait_seconds)

    return f"Service unavailable after {max_attempts} attempts. Please try again later."


retry_agent = Agent(
    name="RetryAgent",
    instructions=(
        "Fetch information using the available tool. "
        "Report only the returned result. "
        "Do not offer follow-up actions."
    ),
    model=MODEL,
    tools=[fetch_with_retry]
)

# --------------------------------------------------------------
print("✅ Retry agent ready")

#### Run with Retry

In [ ]:
random.seed(1)  # reproducible demo: fails once, then succeeds

print("🔁 RETRY PATTERN DEMO")
print("=" * 60)
print("Fetching from unreliable service (60% failure rate)...\n")

result = await Runner.run(retry_agent, input="Fetch the latest sales report")

print(f"\n🤖 {result.final_output}")
print("=" * 60)

### 💡 Key Takeaway

Keep retries inside the tool: the agent never knows a retry happened.

⚠️ **Safety note:** Retry only operations safe to repeat.

An idempotent operation leaves the system in the same state whether it runs once or many times.

A failed response does not prove the write failed.

Retrying orders, messages, or record inserts may duplicate side effects.

---

## 💪 Practice Exercise

### Robust Calculator

*Covers: structured tool arguments and crash-safe error handling*

Build a calculator tool that handles division by zero, invalid inputs, and overflow gracefully.

In [ ]:
# --------------------------------------------------------------
# 💪 Robust Calculator
# --------------------------------------------------------------
# Objective: Handle calculation errors.
#            Return a clear message for each case.

# TODO 1: Create a @function_tool called safe_calculate
#            Parameters: a (float), operator (str), b (float)
#            Print a marker inside the tool so you can see it run, e.g.
#              print(f"  🔧 [tool] safe_calculate({a} {operator} {b})")
#            Allow only these operators: +, -, *, /, **
#            Return a clear error message for any other operator
#            Catch: ZeroDivisionError and OverflowError
#            Add a final broad except Exception fallback that logs the
#            details and returns a safe message (never eval free-form text)

# TODO 2: Create an agent that uses safe_calculate
#            Instruct it to always use the tool for calculations

# TODO 3: Test with these inputs:
#            - "What is 144 / 12?"
#            - "What is 10 / 0?"
#            - "What is 10.0 to the power of 500?"
#            Print each result and confirm the tool marker appears for every test

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Choose how function-tool failures reach the model:**

- By default, the SDK returns a standard error result

- `failure_error_function=None` stops the run with a `UserError`

- Catch expected failures when the model needs domain-specific recovery

- Catch specific exceptions like `KeyError` or `ConnectionError` before a broad `except Exception`

- Log unexpected details and return a safe fallback
<br>
<br>

**Retry transient failures:**

- Set a `max_attempts` limit, never retry without a ceiling

- Add a short `time.sleep()` between attempts to avoid hammering a struggling service

- Return a clear failure message if all attempts are exhausted

- Retry only read-only or safe-to-repeat actions

---

## 📍 Next Step

**Notebook 07: Testing & Evaluating Agents**  

Build a golden test set, measure what's actually working, and catch regressions early.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-06-error-handling-and-recovery)

---